# The demo: pandas, Polars and Arrow

The query: add the station name and altitude to each measurement (join), keep one station (filter), compute the monthly mean temperature.
The join comes before the filter on purpose. pandas runs it in that order, Polars reorders it.

In [1]:
import statistics
import time
from pathlib import Path

import pandas as pd
import polars as pl
import pyarrow as pa

DATA = Path("data")  # the data/ folder next to this notebook
MEASUREMENTS_FILE = DATA / "measurements.parquet"
STATIONS_FILE = DATA / "stations.parquet"
STATION = "ber"  # Bern / Zollikofen
REPS = 10  # each query is timed 10 times

print(f"pandas {pd.__version__}, polars {pl.__version__}, pyarrow {pa.__version__}")
rows = pl.scan_parquet(MEASUREMENTS_FILE).select(pl.len()).collect().item()  # counts the rows from the file metadata
print(f"{MEASUREMENTS_FILE.stat().st_size / 1e6:.0f} MB Parquet, {rows:,} rows")

pandas 3.0.6, polars 1.44.2, pyarrow 25.0.1
31 MB Parquet, 3,831,715 rows


## 1. pandas, eager

### pandas: the query

In [2]:
# monthly mean temperature of a DataFrame
def monthly_mean(df):
    df = df.assign(month=df["ts"].dt.strftime("%Y-%m"))  # "2026-03", "2026-04", ...
    return df.groupby(["month", "station_name", "height_masl"], as_index=False)["temperature"].mean()


# pandas eager: each line runs right away, in the order written
def run_pandas():
    measurements = pd.read_parquet(MEASUREMENTS_FILE)  # reads the whole file
    stations = pd.read_parquet(STATIONS_FILE)
    joined = measurements.merge(stations, on="station_abbr")  # joins every station
    one_station = joined[joined["station_abbr"] == STATION]  # keeps one station, the rest is thrown away
    return monthly_mean(one_station)

### pandas: run it

In [3]:
start = time.perf_counter()
result_pandas = run_pandas()
print(f"{len(result_pandas)} rows in {time.perf_counter() - start:.2f} s")

# rebuild the joined table to measure its size in memory (every station, every column)
joined = pd.read_parquet(MEASUREMENTS_FILE).merge(pd.read_parquet(STATIONS_FILE), on="station_abbr")
print(f"joined table in memory: {joined.memory_usage(deep=True).sum() / 1e6:.0f} MB")
del joined  # free the memory

result_pandas

7 rows in 1.49 s
joined table in memory: 305 MB


,month,station_name,height_masl,temperature
0,2026-03,Bern / Zollikofen,553,3.081081
1,2026-04,Bern / Zollikofen,553,11.273125
2,2026-05,Bern / Zollikofen,553,14.511514
3,2026-06,Bern / Zollikofen,553,20.758843
4,2026-07,Bern / Zollikofen,553,22.804211
5,2026-08,Bern / Zollikofen,553,22.586223
6,2026-09,Bern / Zollikofen,553,17.340459


## 2. Polars, lazy

### Polars, not optimised: the plan as written

In [4]:
# Polars lazy: scan_parquet reads nothing, each step is only added to a plan
def plan_polars():
    measurements = pl.scan_parquet(MEASUREMENTS_FILE)
    stations = pl.scan_parquet(STATIONS_FILE)
    return (
        measurements.join(stations, on="station_abbr")
        .filter(pl.col("station_abbr") == STATION)  # same order as pandas
        .group_by(pl.col("ts").dt.strftime("%Y-%m").alias("month"), "station_name", "height_masl")
        .agg(pl.col("temperature").mean())
        .sort("month")
    )


plan = plan_polars()  # a LazyFrame: the plan, no data yet
print(plan.explain(optimized=False))  # the plan as written

SORT BY [col("month")]
  AGGREGATE[maintain_order: false]
    [col("temperature").mean()] BY [col("ts").dt.to_string().alias("month"), col("station_name"), col("height_masl")]
    FROM
    FILTER col("station_abbr") == "ber"
    FROM
      INNER JOIN:
      LEFT PLAN ON: [col("station_abbr")]
        Parquet SCAN [data/measurements.parquet]
        PROJECT */5 COLUMNS
        ESTIMATED ROWS: 3831715
      RIGHT PLAN ON: [col("station_abbr")]
        Parquet SCAN [data/stations.parquet]
        PROJECT */4 COLUMNS
        ESTIMATED ROWS: 149
      END INNER JOIN


### Polars, optimised: the plan that runs

In [5]:
print(plan.explain())  # the plan Polars runs, after optimisation
print("\n-> the FILTER moved into the Parquet scan (SELECTION), before the JOIN")

SORT BY [col("month")]
  AGGREGATE[maintain_order: false]
    [col("temperature").mean()] BY [col("ts").dt.to_string().alias("month"), col("station_name"), col("height_masl")]
    FROM
    simple π 4/4 ["ts", "station_name", ... 2 other columns]
      INNER JOIN:
      LEFT PLAN ON: [col("station_abbr")]
        Parquet SCAN [data/measurements.parquet]
        PROJECT 3/5 COLUMNS
        SELECTION: col("station_abbr") == "ber"
        ESTIMATED ROWS: 3831715
      RIGHT PLAN ON: [col("station_abbr")]
        Parquet SCAN [data/stations.parquet]
        PROJECT 3/4 COLUMNS
        SELECTION: col("station_abbr") == "ber"
        ESTIMATED ROWS: 149
      END INNER JOIN

-> the FILTER moved into the Parquet scan (SELECTION), before the JOIN


### Polars: run it with collect()

In [6]:
start = time.perf_counter()
result_polars = plan.collect()  # only now Polars reads the file and runs the plan
print(f"{result_polars.height} rows in {time.perf_counter() - start:.2f} s")

result_polars

7 rows in 0.01 s


month,station_name,height_masl,temperature
str,str,i64,f64
"""2026-03""","""Bern / Zollikofen""",553,3.081081
"""2026-04""","""Bern / Zollikofen""",553,11.273125
"""2026-05""","""Bern / Zollikofen""",553,14.511514
"""2026-06""","""Bern / Zollikofen""",553,20.758843
"""2026-07""","""Bern / Zollikofen""",553,22.804211
"""2026-08""","""Bern / Zollikofen""",553,22.586223
"""2026-09""","""Bern / Zollikofen""",553,17.340459


## 3. Timing: median of 10 runs

### The timing function

In [7]:
# run the query REPS times, return the median time, the best time and the result
def time_it(query):
    times = []
    for _ in range(REPS):
        start = time.perf_counter()
        result = query()
        times.append(time.perf_counter() - start)
    return statistics.median(times), min(times), result

### pandas tuned by hand

In [8]:
# pandas written by hand like the Polars plan: 3 columns, one station
def run_pandas_tuned():
    measurements = pd.read_parquet(MEASUREMENTS_FILE, columns=["station_abbr", "ts", "temperature"],
                                   filters=[("station_abbr", "==", STATION)])
    stations = pd.read_parquet(STATIONS_FILE)
    one_station = measurements.merge(stations, on="station_abbr")  # one station joined, not all of them
    return monthly_mean(one_station)

### Compare the three

In [9]:
median_naive, best_naive, result_naive = time_it(run_pandas)
median_tuned, best_tuned, result_tuned = time_it(run_pandas_tuned)
median_lazy, best_lazy, result_lazy = time_it(lambda: plan_polars().collect())

print(f"pandas naive  {median_naive:.3f} s   best {best_naive:.3f} s")
print(f"pandas tuned  {median_tuned:.3f} s   best {best_tuned:.3f} s")
print(f"Polars lazy   {median_lazy:.3f} s   best {best_lazy:.3f} s")
print()

# the three must give the same table
same = pl.from_pandas(result_naive).equals(result_lazy) and pl.from_pandas(result_tuned).equals(result_lazy)
print("same answer from all three:", same)
print(f"tuning pandas by hand: {median_naive / median_tuned:.0f}x faster")
print(f"Polars: {median_tuned / median_lazy:.0f}x faster again")

pandas naive  1.674 s   best 1.481 s
pandas tuned  0.160 s   best 0.138 s
Polars lazy   0.013 s   best 0.012 s

same answer from all three: True
tuning pandas by hand: 10x faster
Polars: 12x faster again


## 4. Arrow: the same memory under Polars and pandas

### One station in Polars
I look at where the temperatures are in memory.

In [13]:
# 1. the data: one station, in one memory block
polars_df = pl.read_parquet(MEASUREMENTS_FILE).filter(pl.col("station_abbr") == STATION)
polars_df = polars_df.rechunk()  # merge the pieces (chunks) into one block
print(f"Polars DataFrame: {polars_df.height:,} rows, {polars_df.estimated_size('mb'):.1f} MB")


# 2. find the memory address where the values of a column start
def buffer_address(column):
    chunk = column.chunks[0]
    values = chunk.buffers()[1]  # its values ([0] would be the null flags)
    return hex(values.address)  # where they start in memory


# 3. example: where the temperatures of this station are
# to_arrow() gives the same table as a pyarrow table, without copying it:
# buffer_address needs .chunks and .buffers(), which a Polars column does not have
print(buffer_address(polars_df.to_arrow()["temperature"]))

Polars DataFrame: 25,788 rows, 0.9 MB
2285231177728


### Passing the table: same address = no copy

I pass the table from Polars to pandas and back, through Arrow.
At each step, I print the address of the temperatures.
Same address = no copy.
pandas with NumPy columns (its default) copies the values.

In [11]:
# 1. Polars -> Arrow
arrow_table = polars_df.to_arrow()
addr_start = buffer_address(arrow_table["temperature"])
print(f"Polars -> Arrow           {addr_start}")

# 2. Arrow -> pandas, with Arrow columns
pandas_arrow = arrow_table.to_pandas(types_mapper=pd.ArrowDtype)
addr = buffer_address(pandas_arrow["temperature"].array._pa_array)  # the Arrow array inside the pandas column
print(f"Arrow -> pandas (Arrow)   {addr}   same: {addr == addr_start}")

# 3. Arrow -> pandas, with NumPy columns (the pandas default)
pandas_numpy = arrow_table.to_pandas()
addr = hex(pandas_numpy["temperature"].to_numpy().ctypes.data)  # memory address of the NumPy array
print(f"Arrow -> pandas (NumPy)   {addr}   same: {addr == addr_start}")

# 4. pandas -> Polars
polars_again = pl.from_pandas(pandas_arrow)
addr = buffer_address(polars_again.rechunk().to_arrow()["temperature"])
print(f"pandas -> Polars          {addr}   same: {addr == addr_start}")

Polars -> Arrow           0x21404880000
Arrow -> pandas (Arrow)   0x21404880000   same: True
Arrow -> pandas (NumPy)   0x21447b80000   same: False
pandas -> Polars          0x21404880000   same: True


### How long does Polars -> Arrow -> pandas -> Polars take?

I time the conversion on the whole table, with and without the text column. Median of 10 runs.

In [12]:
# Polars -> Arrow -> pandas with Arrow columns -> back to Polars
def convert_via_arrow(df):
    return pl.from_pandas(df.to_arrow().to_pandas(types_mapper=pd.ArrowDtype))

# Polars -> Arrow -> pandas with NumPy columns -> back to Polars
def convert_via_numpy(df):
    return pl.from_pandas(df.to_arrow().to_pandas())  # pandas default

full_df = pl.read_parquet(MEASUREMENTS_FILE)
numbers_df = full_df.drop("station_abbr")  # the same table without the text column

# median time of each conversion
arrow_full, _, _ = time_it(lambda: convert_via_arrow(full_df))
numpy_full, _, _ = time_it(lambda: convert_via_numpy(full_df))
arrow_numbers, _, _ = time_it(lambda: convert_via_arrow(numbers_df))
numpy_numbers, _, _ = time_it(lambda: convert_via_numpy(numbers_df))

print(f"                 whole table    without text")
print(f"Arrow columns: {arrow_full * 1e3:8.1f} ms    {arrow_numbers * 1e3:8.1f} ms")
print(f"NumPy columns: {numpy_full * 1e3:8.1f} ms    {numpy_numbers * 1e3:8.1f} ms")
print(f"copy costs     {numpy_full / arrow_full:8.1f}x     {numpy_numbers / arrow_numbers:8.1f}x")

                 whole table    without text
Arrow columns:    177.3 ms        32.8 ms
NumPy columns:    248.1 ms        76.7 ms
copy costs          1.4x          2.3x
